# Reproduce the figures on corporatedisclosuredata.org

This notebook rebuilds every example chart on the home page of
[corporatedisclosuredata.org](https://corporatedisclosuredata.org): the network of companies that share
directors, the company comparison, and the yearly charts on women on boards, audit fees, governance
policies, classified boards, meeting fees and director skills.

It starts from the same raw model extraction file anyone can download from the site (one zip, about 830 MB)
and adds three things a proxy statement does not print, all taken from public SEC data and stored in this
repository: industry codes, stock tickers and public float.

**How long it takes.** In Google Colab: about 5 to 10 minutes the first time (most of it downloading the
raw model extraction file and reading it once), and well under a minute when you run it again, because the
download and a slim extract are kept in the `cdd_cache` folder.

**How to use it.** Run the cells from top to bottom (in Colab: *Runtime > Run all*). The first cell under
*Get the data* asks for your details before the download.

In [ ]:
# Setup: find this repository (in Colab, download it), and install duckdb if it is missing.
import importlib.util, os, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/njhallman/cdd-examples"
IN_COLAB = "google.colab" in sys.modules

here = Path.cwd()
root = next((p for p in [here, *here.parents] if (p / "scripts/site/build_example_data.py").exists()), None)
if root is None:
    if not IN_COLAB:
        raise SystemExit(f"Run this notebook from a copy of the repository: git clone {REPO_URL}")
    subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, "/content/cdd-examples"], check=True)
    root = Path("/content/cdd-examples")
os.chdir(root)
for p in (root / "scripts" / "site", root / "scripts"):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
if importlib.util.find_spec("duckdb") is None:      # reads the two small Parquet files from the SEC
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
print("Working in", root.name)

## Get the data

The raw model extraction is free. Before the download the site asks the same questions as the download form at
[corporatedisclosuredata.org/#download](https://corporatedisclosuredata.org/#download): your name, email,
institution, role and how you will use the data. Your answers are sent to corporatedisclosuredata.org,
which sends back a download link that works for 24 hours; no email is sent.

Fill in the form below (in Colab), or leave it empty and the next cell will ask each question in turn.
The data are licensed under the
[Creative Commons Attribution-NonCommercial 4.0 International license (CC BY-NC 4.0)](https://corporatedisclosuredata.org/data/LICENSE).
Academic research, teaching and other non-commercial use are free, with credit to the dataset. Journalism is
also free, including at commercial news organizations; we grant that permission in addition to the license.
Any other commercial use needs our written permission first; contact the team listed on the Team page of
https://corporatedisclosuredata.org.

**Already have the file?** Set `RAW_PATH` to `raw_extractions.jsonl.gz` or to `cdd_raw_v1.0.0.zip`, or set
`CDD_DOWNLOAD_URL` to the link from an earlier request or email, and no new request is sent. In Colab you
can also set `UPLOAD = True` to upload the file from your computer.

In [ ]:
#@title Your details { display-mode: "form" }
NAME = ""  #@param {type:"string"}
EMAIL = ""  #@param {type:"string"}
INSTITUTION = ""  #@param {type:"string"}
ROLE = ""  #@param ["", "Faculty", "Doctoral student", "Other student", "Journalist", "Other"]
USE = ""  #@param ["", "Research", "Teaching", "Journalism", "Other"]
USE_NOTE = ""  #@param {type:"string"}
HEARD_ABOUT_US = ""  #@param {type:"string"}
EMAIL_ME_ABOUT_NEW_VERSIONS = False  #@param {type:"boolean"}
I_AGREE_TO_THE_DATA_LICENSE = False  #@param {type:"boolean"}

# Skip the request when you already have the file or a link (or set these as environment variables).
RAW_PATH = os.environ.get("CDD_RAW_PATH", "")
CDD_DOWNLOAD_URL = os.environ.get("CDD_DOWNLOAD_URL", "")
UPLOAD = False
BASE = os.environ.get("CDD_BASE", "https://corporatedisclosuredata.org")   # the site that answers the request

In [ ]:
import cdd_download as dl

def ask():
    """Collect the answers: the form values above, or a typed answer for each one left empty."""
    print("The same questions as the site's download form; the answers are sent to corporatedisclosuredata.org.\n")
    def text(value, prompt):
        return value.strip() or input(prompt).strip()
    def pick(value, options, prompt):
        if value:
            return value
        print(prompt)
        for i, label in enumerate(options.values(), 1):
            print(f"  {i}. {label}")
        return input("Type a number: ").strip()
    answers = {
        "name": text(NAME, "Your name: "),
        "email": text(EMAIL, "Your email: "),
        "institution": text(INSTITUTION, "Your institution (university or organization; optional): "),
        "role": pick(ROLE, dl.ROLES, "Your role:"),
        "use": pick(USE, dl.USES, "How will you use the data?"),
    }
    answers["use_detail"] = USE_NOTE.strip() or input(
        "Say more about how you will use the data" + (": " if dl.choice(answers["use"], dl.USES, "use") == "other" else " (optional): ")).strip()
    answers["heard"] = HEARD_ABOUT_US.strip()
    answers["news"] = EMAIL_ME_ABOUT_NEW_VERSIONS
    agree = I_AGREE_TO_THE_DATA_LICENSE or input(
        "Do you agree to the Creative Commons Attribution-NonCommercial 4.0 International license (CC BY-NC 4.0), "
        "https://corporatedisclosuredata.org/data/LICENSE ? [y/N] ").strip().lower() in ("y", "yes")
    answers["agree"] = bool(agree)
    return answers

if UPLOAD and IN_COLAB and not RAW_PATH:
    from google.colab import files
    RAW_PATH = next(iter(files.upload()))

try:
    RAW = dl.get_raw("cdd_cache", raw_path=RAW_PATH, download_url=CDD_DOWNLOAD_URL, base=BASE, ask=ask)
except dl.DownloadError as e:
    raise SystemExit(f"Could not get the raw model extraction file. {e}") from None

## Build the chart data

This runs `scripts/site/build_example_data.py`, the same script that builds the charts on the site, on the
raw model extraction file and the SEC files in `data/` and `site/`. It reads that file once (about a minute)
and keeps a slim extract in `cdd_cache`, so later runs take seconds. The result is three Python objects with
the same names the site's charts read: `D` (the yearly series), `C` (the 2025 companies and the network) and `SK`
(director skills).

The SEC files in this repository:

* `data/sic/firm_sic.parquet`: the industry (SIC) code SEC EDGAR lists for each company;
* `site/data/filings_2025.parquet`: stock ticker and exchange, from the SEC's ticker file;
* `data/site/public_float.csv`: every public float each company reported to the SEC, after the checks in
  `scripts/site/fetch_public_float.py` and the hand-checked list in `data/site/public_float_exclusions.csv`
  (dropped values are listed in `data/site/public_float_dropped.csv`);
* `data/site/sec_entity_names.csv` and `site/company_display.js`: the SEC's company names and the site's brand
  spellings, used only to write company names in normal capitals.

The definitions of every number are at the top of `scripts/site/build_example_data.py`.

In [ ]:
# Optional: download public float from SEC EDGAR again instead of using data/site/public_float.csv.
# Takes about 35 minutes (the SEC allows about 8 requests a second). The SEC keeps adding filings, so the
# results can differ slightly from the site. Left off by default.
REFETCH_PUBLIC_FLOAT = False

PUBLIC_FLOAT_CSV = "data/site/public_float.csv"
ENTITY_NAMES_CSV = "data/site/sec_entity_names.csv"
if REFETCH_PUBLIC_FLOAT:
    contact = input("The SEC asks automated downloads to name a contact email (sent only to sec.gov): ").strip()
    out = Path("cdd_cache/refetch")
    subprocess.run([sys.executable, "scripts/site/fetch_public_float.py", "--raw", str(RAW),
                    "--cache", str(out / "responses"), "--out", str(out / "public_float.csv"),
                    "--names-out", str(out / "sec_entity_names.csv"),
                    "--dropped-out", str(out / "public_float_dropped.csv"),
                    "--user-agent", f"cdd-examples notebook {contact}"], check=True)
    PUBLIC_FLOAT_CSV, ENTITY_NAMES_CSV = str(out / "public_float.csv"), str(out / "sec_entity_names.csv")

In [ ]:
import time, types
import build_example_data as bed

inputs = types.SimpleNamespace(
    firm_sic="data/sic/firm_sic.parquet",
    filings_2025="site/data/filings_2025.parquet",
    public_float=PUBLIC_FLOAT_CSV,
    entity_names=ENTITY_NAMES_CSV,
    display_js="site/company_display.js",
)
Path("cdd_cache").mkdir(exist_ok=True)
start = time.time()
D, C, SK, extra = bed.build(str(RAW), "cdd_cache/slim_v1.0.0.jsonl.gz", inputs)
JS = bed.render_js(D, C, SK)
Path("cdd_cache/example-data.js").write_text(JS, encoding="utf-8")
t = D["tiles"]
print(f"Built in {time.time() - start:.0f} seconds.")
print(f"{t['filings']:,} proxy statements, {t['companies']:,} companies, {t['director_rows']:,} director listings, "
      f"{t['first_year']} to {t['last_year']}.")

**Check against the site.** The next cell downloads the data behind the site's charts,
[example-data.js](https://corporatedisclosuredata.org/example-data.js), and compares every series with the
one just built. Without an internet connection it says so and moves on.

In [ ]:
import json, re, urllib.request

EXPECTED_JS = os.environ.get("CDD_EXPECTED_JS", BASE.rstrip("/") + "/example-data.js")

def parse_js(text):
    return {m.group(1): json.loads(m.group(2)) for m in re.finditer(r"^const (\w+) = (.*);$", text, re.M)}

def leaves(obj, path):
    """Each series: a dict is opened up when any of its values is itself a dict."""
    if isinstance(obj, dict) and any(isinstance(v, dict) for v in obj.values()):
        for k, v in obj.items():
            yield from leaves(v, f"{path}.{k}")
    else:
        yield path, obj

try:
    if EXPECTED_JS.startswith("http"):
        with urllib.request.urlopen(urllib.request.Request(EXPECTED_JS, headers={"user-agent": dl.USER_AGENT}), timeout=60) as r:
            published_text = r.read().decode("utf-8")
    else:
        published_text = Path(EXPECTED_JS).read_text(encoding="utf-8")
except Exception as e:
    published_text = None
    print(f"Could not fetch {EXPECTED_JS} ({e}); the comparison is skipped.")

if published_text:
    published = parse_js(published_text)
    mine = {"D": D, "C": C, "SK": SK}
    ours = dict(p for name in mine for p in leaves(mine[name], name))
    theirs = dict(p for name in published for p in leaves(published[name], name))
    differ = [k for k in sorted(set(ours) | set(theirs)) if ours.get(k) != theirs.get(k)]
    if differ:
        print(f"{len(differ)} of {len(theirs)} series differ from the site:")
        for k in differ:
            print("  ", k)
    else:
        print(f"All {len(theirs)} series match the site's example-data.js.")
    print("The file is byte-for-byte identical." if JS == published_text else
          "The files are not byte-for-byte identical.")

**Chart settings.** Colours, fonts and two small helpers shared by the charts below. Each chart is shown here
and saved to the `figures` folder.

In [ ]:
import math, random, statistics, textwrap
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

# The site's colours (light theme).
INK, INK2, MUTED, PAPER = "#0f1722", "#2b3440", "#4a5665", "#ffffff"
GRID, AXIS = "#e7ebf0", "#7a8594"
S = ["#14305c", "#bf5700", "#1a8a8a", "#b8860b", "#7a4fa3", "#3a8f4b", "#c0392b", "#6b7280"]          # --s1 .. --s8
B = [None, "#0c2445", "#223857", "#384d6a", "#4f627e", "#677891", "#7f8fa5", "#99a7b9", "#b3bfce", "#ced7e3", "#eaf0f8"]  # --b1 .. --b10
SECTORS = bed.SECTORS
SECTOR_COLOR = {s: S[i] for i, s in enumerate(SECTORS)}
PARTIAL = 2026                                    # the last year in the data is not complete
SOURCE = "Source: corporatedisclosuredata.org, raw model extraction, version 1.0.0."

plt.rcParams.update({
    "figure.facecolor": PAPER, "axes.facecolor": PAPER, "savefig.facecolor": PAPER,
    "font.size": 11, "axes.titlesize": 15, "axes.titleweight": "bold", "axes.titlelocation": "left",
    "axes.edgecolor": AXIS, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "axes.spines.left": False,
    "axes.grid": True, "axes.grid.axis": "y", "grid.color": GRID, "grid.linewidth": 1,
    "legend.frameon": False,
    "text.parse_math": False,      # so "$2B to $10B" prints as written
})
FIGURES = Path("figures")
FIGURES.mkdir(exist_ok=True)


def titles(fig, title, subtitle):
    fig.suptitle(title, x=0.01, y=0.995, ha="left", va="top", fontsize=16, fontweight="bold")
    fig.text(0.01, 0.935, "\n".join(textwrap.wrap(subtitle, 105)), ha="left", va="top", fontsize=11, color=INK2)


def finish(fig, name, note=""):
    """Add the source line, save figures/<name>.png and show the chart."""
    text = (note + " " if note else "") + SOURCE
    fig.text(0.01, 0.0, "\n".join(textwrap.wrap(text, 140)), ha="left", va="top", fontsize=8.5, color=MUTED)
    path = FIGURES / f"{name}.png"
    fig.savefig(path, dpi=200, bbox_inches="tight", pad_inches=0.2)
    plt.show()
    print("Saved", path)


def pct(v, _=None):
    return f"{v * 100:.0f}%"


def usd(v, _=None):
    if v == 0:
        return "0"
    return f"${v / 1e6:.{0 if v >= 1e7 else 1}f}M".replace(".0M", "M") if v >= 1e6 else f"${v / 1e3:.0f}k"


def nice_max(v, is_pct):
    if is_pct:
        return next((x for x in (0.1, 0.2, 0.25, 0.3, 0.4, 0.5, 0.6, 0.8, 1) if x >= v * 1.04), 1)
    e = 10 ** math.floor(math.log10(v))
    m = v / e
    return (1 if m <= 1 else 2 if m <= 2 else 2.5 if m <= 2.5 else 5 if m <= 5 else 10) * e


def line_chart(name, title, subtitle, series, start, fmt=pct, ymax=None, annotations=(), label_ends=False,
               legend=False, note=""):
    """Yearly lines as on the site: the last, partial year is a dotted segment ending in an open circle.

    series: (label, colour, {year: value}). annotations: (year, text, side, row), side "right" or "left".
    """
    fig, ax = plt.subplots(figsize=(10, 5.6))
    fig.subplots_adjust(top=0.78 if annotations else 0.83, right=0.74 if label_ends else 0.97, left=0.08, bottom=0.09)
    titles(fig, title, subtitle)
    data = [(label, colour, sorted((int(y), v) for y, v in values.items() if int(y) >= start and v is not None))
            for label, colour, values in series]
    top = ymax or nice_max(max(v for _, _, d in data for _, v in d), fmt is pct)
    x0 = min(y for _, _, d in data for y, _ in d)
    x1 = max(y for _, _, d in data for y, _ in d)
    for label, colour, d in data:
        solid = [(y, v) for y, v in d if y < PARTIAL]
        ax.plot(*zip(*solid), color=colour, lw=2.2, solid_capstyle="round", label=label)
        part = [(y, v) for y, v in d if y >= PARTIAL - 1]
        if len(part) == 2 and part[1][0] == PARTIAL:
            ax.plot(*zip(*part), color=colour, lw=2.2, ls=(0, (1, 2)))
            ax.plot([PARTIAL], [part[1][1]], "o", ms=7, mfc=PAPER, mec=colour, mew=2)
    for year, text, side, row in annotations:
        ax.axvline(year, color=AXIS, lw=1, ls=(0, (3, 3)), zorder=0)
        ax.annotate(text, xy=(year, 1), xycoords=("data", "axes fraction"),
                    xytext=(-5 if side == "left" else 5, 6 + 18 * row), textcoords="offset points",
                    ha="right" if side == "left" else "left", va="bottom", fontsize=11, color=INK)
    if label_ends:                       # names at the right end, nudged apart so they do not overlap
        ends = sorted(((d[-1][1], label) for label, _, d in data), reverse=True)
        gap = top * 0.06
        placed = []
        for v, label in ends:
            y = v if not placed or placed[-1] - v >= gap else placed[-1] - gap
            placed.append(y)
            ax.annotate(label, xy=(x1, v), xytext=(x1 + (x1 - x0) * 0.035, y), textcoords="data",
                        va="center", fontsize=12, fontweight="bold", color=INK, annotation_clip=False)
    if legend:
        ax.legend(loc="upper left", bbox_to_anchor=(0, 1.0), ncol=len(data), fontsize=11, handlelength=1.2)
    ax.set_xlim(x0 - 0.3, x1 + 0.5)
    ax.set_ylim(0, top)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(nbins=6, steps=[1, 2, 2.5, 5, 10]))
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(fmt))
    ax.xaxis.set_major_locator(mticker.MultipleLocator(5))
    ax.tick_params(length=0, labelsize=11)
    finish(fig, name, (("The last point (open circle) covers proxy statements filed so far in 2026. " if x1 == PARTIAL else "") + note).strip())

## Network

*Corporate Board Interlock Network.* Each dot is a company with a 2025 annual-meeting proxy statement. A
line joins two companies when the same director sits on both boards.

A line joins two companies when a director with the same first and last name, middle names or initials that
do not disagree, and ages within one year of each other, sits on both boards. A director whose age is not in
the proxy statement is not matched, so some real links are missed, and a few different people who share a
name and age are joined. Departing directors are left out. Colours show industry, which comes from SEC
industry (SIC) codes, not from the proxy statement. Bigger dots have more links.

In [ ]:
nodes, edges = C["net"]["nodes"], C["net"]["edges"]      # node: x, y, company, industry, number of links
fig, ax = plt.subplots(figsize=(11, 7.6))
fig.subplots_adjust(top=0.86, bottom=0.08, left=0.01, right=0.99)
titles(fig, "Corporate Board Interlock Network",
       f"Graph displays data from {C['net']['n_firms']:,} companies’ 2025 annual-meeting proxy statements, "
       f"{C['net']['linked']:,} of which share a director with another firm in the sample. The chart shows the "
       f"{len(nodes):,} of them that are each linked to at least two others in the chart, joined by {len(edges):,} lines.")
ax.add_collection(LineCollection([[(nodes[a][0], nodes[a][1]), (nodes[b][0], nodes[b][1])] for a, b in edges],
                                 colors=INK, linewidths=0.4, alpha=0.13, zorder=1))
order = sorted(range(len(nodes)), key=lambda i: nodes[i][4])       # well-linked companies on top
radius = [min(7, 1.1 + math.sqrt(nodes[i][4]) * 0.62) for i in order]
ax.scatter([nodes[i][0] for i in order], [nodes[i][1] for i in order], s=[(1.5 * r) ** 2 for r in radius],
           c=[SECTOR_COLOR.get(nodes[i][3], MUTED) for i in order], alpha=0.92, linewidths=0.5, edgecolors=PAPER, zorder=2)
ax.set_xlim(-0.02, 1.02)
ax.set_ylim(1.02, -0.02)          # the site draws y downwards
ax.axis("off")
handles = [Line2D([], [], ls="", marker="o", ms=8, mfc=SECTOR_COLOR[s], mec=PAPER, label=s) for s in SECTORS]
if any(n[3] is None for n in nodes):
    handles.append(Line2D([], [], ls="", marker="o", ms=8, mfc=MUTED, mec=PAPER, label="No industry code"))
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, 0.0), ncol=5, fontsize=10, handletextpad=0.2)
finish(fig, "network", "Industry from SEC industry (SIC) codes. Positions come from a seeded force-directed layout.")

## Compare companies

Compare companies in their 2025 proxy statements. The site lets you choose a measure and a way to group
companies; here `compare(measure, by)` does the same. The first chart is the site's default: audit fees by
public float.

One row for each 2025 company, from its latest 2025 annual-meeting proxy statement. Board size, women on the
board and director age count every director the proxy statement lists. A company's audit fees are the total
its proxy statement prints, or the sum of its fee lines when it prints none. Public float is the market value
of shares held by investors outside the company; it is not printed in a proxy statement, so we use the latest
figure the company reported to the SEC within 400 days before the proxy statement. The cut points are the
usual large-, mid-, small- and micro-cap thresholds.

Measures: `"Audit fees"`, `"Women on the board"`, `"Board size"`, `"Director age"`. Groupings: `"Public float"`,
`"Auditor"`, `"First proxy year"`, `"Industry"`, `"Share classes"`.

In [ ]:
F = C["firms"]   # name, ticker, industry, share women, audit fee, median age, directors, (unused), women,
                 # exchange, auditor, fee quarter, first proxy year group, share classes, public float group
CUTS = {
    "Public float": (14, ["$10B or more", "$2B to $10B", "$300M to $2B", "Under $300M"], True,
                     lambda g: "Companies with a public float of " + ("under $300M" if g == "Under $300M" else g)),
    "Auditor": (10, ["PwC", "EY", "Deloitte", "KPMG", "Grant Thornton", "BDO", "RSM", "Other firms"], False,
                lambda g: "Companies audited by " + ("other firms" if g == "Other firms" else g)),
    "First proxy year": (12, ["2000 or earlier", "2001 to 2010", "2011 to 2020", "2021 or later"], True,
                         lambda g: "Companies first filing in " + g),
    "Industry": (2, SECTORS, False, lambda g: "Companies in " + g.lower()),
    "Share classes": (13, ["Dual-class shares", "Single class of shares"], False,
                      lambda g: "Companies with " + ("dual-class shares" if g == "Dual-class shares" else "a single class of shares")),
}
MEASURES = {
    "Audit fees": dict(key=4, kind="swarm", by="Public float", lo=3e4, hi=1e8, ticks=[1e5, 1e6, 1e7, 1e8], fmt=usd),
    "Women on the board": dict(key=8, kind="stack", by="Public float", cats=["0", "1", "2", "3", "4", "5 or more"]),
    "Board size": dict(key=6, kind="hist", by="Public float", lo=3, hi=18, ticks=[3, 6, 9, 12, 15, 18], top="18+"),
    "Director age": dict(key=5, kind="hist", by="First proxy year", lo=45, hi=75, ticks=[45, 50, 55, 60, 65, 70, 75],
                         bot="45 or under", top="75+"),
}
ORDINAL = [B[2], B[4], B[6], B[8]]
RAMP = [B[10], B[8], B[6], B[4], B[2], B[1]]


def compare(measure="Audit fees", by=None, save_as=None):
    """Draw one measure for the 2025 companies, one row per group. Saved to figures/<save_as>.png if given."""
    m = MEASURES[measure]
    by = by or m["by"]
    key, groups, ordinal, describe = CUTS[by]
    colour = lambda i, g: ORDINAL[i] if ordinal else SECTOR_COLOR[g] if by == "Industry" else S[i]
    rows = [(g, [f for f in F if f[key] == g and f[m["key"]] is not None]) for g in groups]
    rows = [(g, fs) for g, fs in rows if fs]
    band = {"swarm": 1.0, "hist": 1.0, "stack": 0.62}[m["kind"]]
    fig, axes = plt.subplots(len(rows), 1, figsize=(10, 1.15 + band * 1.05 * len(rows)), sharex=True, squeeze=False)
    axes = axes[:, 0]
    fig.subplots_adjust(top=1 - 0.95 / fig.get_figheight(), bottom=0.5 / fig.get_figheight(), hspace=0.75, left=0.02, right=0.98)
    fig.suptitle(f"{measure} in the 2025 proxy statements, by {by.lower()}", x=0.02, y=0.995, ha="left", va="top",
                 fontsize=15, fontweight="bold")
    rnd = random.Random(1)
    for i, ((g, fs), ax) in enumerate(zip(rows, axes)):
        values = [f[m["key"]] for f in fs]
        ax.grid(False)
        ax.tick_params(length=0, labelsize=10.5)
        ax.set_yticks([])
        if m["kind"] == "swarm":
            lo, hi = math.log10(m["lo"]), math.log10(m["hi"])
            xs = sorted(min(hi, max(lo, math.log10(v))) for v in values)
            # A simple beeswarm in screen-like units (1000 across, dot radius 2.3), as on the site.
            unit, r, half = 1000 / (hi - lo), 2.3, 34
            d = 2 * r + 0.4
            placed, ys, start = [], [], 0
            for x in xs:
                px = (x - lo) * unit
                while start < len(placed) and placed[start][0] < px - d:
                    start += 1
                y, ok = 0.0, False
                for k in range(400):
                    y = (1 if k % 2 else -1) * math.ceil(k / 2) * r * 0.9
                    if abs(y) > half:
                        break
                    ok = all((qx - px) ** 2 + (qy - y) ** 2 >= d * d for qx, qy in placed[start:])
                    if ok:
                        break
                if not ok:
                    y = (rnd.random() * 2 - 1) * half
                placed.append((px, y))
                ys.append(y)
            ax.scatter(xs, ys, s=8, color=colour(i, g), alpha=0.85, linewidths=0)
            med = math.log10(statistics.median(values))
            ax.plot([med, med], [-half - 3, half + 3], color=INK, lw=2)
            ax.set_ylim(-half - 4, half + 4)
            ax.set_xlim(lo, hi)
            ax.set_xticks([math.log10(t) for t in m["ticks"]], [m["fmt"](t) for t in m["ticks"]])
            for t in m["ticks"]:
                ax.axvline(math.log10(t), color=GRID, lw=1, zorder=0)
            stat = f"median {m['fmt'](statistics.median(values))} · {len(fs):,} companies"
        elif m["kind"] == "hist":
            bins = list(range(m["lo"], m["hi"] + 1))
            counts = [0] * len(bins)
            for v in values:
                counts[min(m["hi"], max(m["lo"], round(v))) - m["lo"]] += 1
            ax.bar(bins, [c / len(values) for c in counts], width=0.8, color=colour(i, g))
            ax.set_xticks(m["ticks"], [m.get("top") if t == m["hi"] and m.get("top") else
                                       ("≤" + str(t)) if t == m["lo"] and m.get("bot") else str(t) for t in m["ticks"]])
            ax.set_xlim(m["lo"] - 0.6, m["hi"] + 0.6)
            ax.spines["bottom"].set_visible(True)
            med = statistics.median(values)
            stat = f"median {round(med * 2) / 2:g} · {len(fs):,} companies"
        else:
            counts = [0] * len(m["cats"])
            for v in values:
                counts[min(5, v)] += 1
            left = 0
            for c, (n, col) in enumerate(zip(counts, RAMP)):
                share = n / len(values)
                ax.barh(0, share, left=left, height=0.8, color=col, edgecolor=PAPER, linewidth=1.5)
                if share > 0.035:
                    ax.text(left + share / 2, 0, f"{share * 100:.0f}%", ha="center", va="center", fontsize=10,
                            color=INK if c < 3 else PAPER)
                left += share
            ax.set_xlim(0, 1)
            ax.set_ylim(-0.45, 0.45)
            ax.tick_params(labelbottom=False)
            stat = f"{(len(values) - counts[0] - counts[1]) / len(values) * 100:.0f}% have two or more women · {len(fs):,} companies"
        ax.set_title(describe(g), loc="left", fontsize=12, fontweight="bold", pad=4)
        ax.text(1, 1.02, stat, transform=ax.transAxes, ha="right", va="bottom", fontsize=10.5, color=INK2)
    if m["kind"] == "stack":
        axes[0].legend(handles=[Patch(color=c, label=l) for c, l in zip(RAMP, m["cats"])], title="Women on the board",
                       loc="lower right", bbox_to_anchor=(1, 1.45), ncol=6, fontsize=9.5, title_fontsize=9.5)
    notes = {"swarm": "Each dot is a company; the dark line is the median. Log scale.",
             "hist": "Bar height is the share of companies in the group.", "stack": "Share of companies with each number of women on the board."}
    missing = sum(1 for f in F if f[m["key"]] is None)
    what = "no usable audit fee total in U.S. dollars" if m["key"] == 4 else "no value"
    note = notes[m["kind"]] + (f" {missing:,} companies with {what} are left out." if missing else "")
    if by == "Public float":
        note += f" {sum(1 for f in F if f[14] is None):,} companies with no public float in the SEC data are not grouped."
    if by == "Industry":
        note += " Industry from SEC industry (SIC) codes."
    if save_as:
        finish(fig, save_as, note)
    else:
        plt.show()


compare("Audit fees", "Public float", save_as="compare_companies")

In [ ]:
# Try other measures and groupings, for example:
compare("Women on the board", "Public float")
compare("Director age", "First proxy year")

## Women on boards

Average share of board seats held by women, each year. For each annual-meeting proxy statement we take the
share of listed directors recorded as women, then average across the proxy statements filed in each year.

In [ ]:
line_chart("women_on_boards", "Women on boards", "Average share of board seats held by women, each year.",
           [("Women", S[0], D["female"])], start=1997, ymax=0.4,
           annotations=[(2017, "Big Three investor campaigns", "left", 0), (2018, "California SB 826", "right", 0),
                        (2021, "Nasdaq diversity rule", "right", 1)])

## Audit fees

Fees paid to each company's auditor, by type of service, each year. A company's total is the total its proxy
statement prints, or the sum of its fee lines when it prints none. The first chart is the median across
annual-meeting proxy statements in each year. The second divides each of the four fee types by their sum, so
each company's shares add to 100%, and averages across companies.

Amounts printed in thousands or millions are converted to dollars; if that gives a total above $200 million,
the units label is taken to be wrong and the amounts are read as dollars. Proxy statements that report fees in
another currency are left out. Fee disclosure in proxy statements began with filings in 2001, and the fee types
were defined differently before 2003; how those early fees map into the four types has not been independently
checked. Dollar amounts are nominal.

In [ ]:
line_chart("audit_fees_total", "Audit fees", "Median total fees paid to the auditor, per company, each year.",
           [("Median total", S[0], D["audit_median_total"])], start=2001, fmt=usd,
           annotations=[(2002, "Sarbanes-Oxley", "right", 0)], note="Nominal dollars.")
mix = D["audit_mix"]
line_chart("audit_fees_mix", "Audit fees by type of service",
           "Average share of each company's fees, by type of service, each year.",
           [("Audit", S[0], mix["audit"]), ("Audit-related", S[2], mix["audit_related"]),
            ("Tax", S[3], mix["tax"]), ("All other", S[1], mix["all_other"])], start=2001, legend=True)

## Governance policies

Share of companies whose proxy statement discloses a clawback policy, a hedging policy, proxy access or a limit
on outside board seats, each year: the share of annual-meeting proxy statements in each year in which each
policy was found. The SEC's clawback listing rule took effect in 2023, and its hedging disclosure rule applied
to proxy statements from 2020.

In [ ]:
p = D["policies"]
line_chart("governance_policies", "Governance policies",
           "Share of companies whose proxy statement discloses each policy, each year.",
           [("Hedging policy", S[0], p["hedging"]), ("Clawback policy", S[1], p["clawback"]),
            ("Outside-board limit", S[2], p["overboarding"]), ("Proxy access", S[3], p["proxy_access"])],
           start=2005, label_ends=True)

## Classified boards

Share of companies with a classified (staggered) board, each year: the top quarter by public float against
all other companies. Whether a board is classified comes from the raw model extraction. Public float, the market value of
shares held outside the company, is not printed in a proxy statement: we use the latest figure the company
reported to the SEC within 400 days before the proxy statement. A few companies report a float a thousand times
too large; we drop any figure far out of line with the same company's other years. Each year the top quarter
is the companies at or above the 75th percentile of float. The chart starts in 2012, when float is available
for enough companies.

In [ ]:
cl = D["classified"]
coverage = [v for y, v in cl["coverage"].items() if int(y) != PARTIAL]
no_float = round(100 * (1 - sum(coverage) / len(coverage)))
line_chart("classified_boards", "Classified boards",
           "Share of companies with a classified (staggered) board: the top quarter by public float against all other companies.",
           [("Top quarter", B[2], cl["top2"]), ("All other companies", B[7], cl["rest"])], start=2012, ymax=0.6,
           label_ends=True,
           note=f"The top quarter is about {cl['n_top_avg']:,} companies a year. On average {no_float}% of annual-meeting "
                "proxy statements a year have no public float and are in neither line. Public float from SEC filings.")

## Meeting fees

Share of director fee schedules that pay a fee for attending board or committee meetings: among
annual-meeting proxy statements that print a director fee schedule, the share whose schedule includes a fee
for attending meetings.

In [ ]:
line_chart("meeting_fees", "Meeting fees",
           "Share of director fee schedules that pay a fee for attending board or committee meetings.",
           [("Pays a meeting fee", S[0], D["meeting_fee"])], start=1997)

## Skills trend

Share of proxy statements listing director skills that name cybersecurity, artificial intelligence, climate
and sustainability, or diversity and inclusion. This is a keyword search of the companies' own skill lists.
Among annual-meeting proxy statements that list skills for at least one director, a proxy statement counts for
a topic when any director's listed skills contain one of its keywords: "cyber", "information security" or
"data security"; "artificial intelligence", "AI" or "machine learning"; "climate", "sustainab", "ESG" or
"environmental"; "diversity", "inclusion" or "DEI".

In [ ]:
tr = SK["trend"]
line_chart("skills_trend", "Director skills",
           "Share of proxy statements listing director skills that name each topic, each year.",
           [("Cybersecurity", S[0], tr["cyber"]), ("Climate and sustainability", S[2], tr["esg"]),
            ("Artificial intelligence", S[1], tr["ai"]), ("Diversity and inclusion", S[6], tr["dei"])],
           start=2010, label_ends=True, annotations=[(2023, "SEC cyber rule", "left", 0)])

## Skills by director

*The skills companies list for their directors.* Share of directors credited with each skill, among directors
whose proxy statement lists skills, in 2019 and 2025. A director counts for a skill when the list contains one
of its keywords, for example "financ" for Finance or "risk" for Risk management (the full lists are in
`SKILL_PATTERNS` in `scripts/site/build_example_data.py`). The gap between the dots is the change from 2019 to
2025.

In [ ]:
rows = SK["dumbbell"]
fig, ax = plt.subplots(figsize=(10, 0.36 * len(rows) + 1.6))
fig.subplots_adjust(left=0.27, right=0.95, top=1 - 1.05 / fig.get_figheight(), bottom=0.45 / fig.get_figheight())
titles(fig, "The skills companies list for their directors",
       "Share of directors credited with each skill, among directors whose proxy statement lists skills.")
ys = range(len(rows))
ax.hlines(ys, [r["y2019"] for r in rows], [r["y2025"] for r in rows], color=S[0], alpha=0.45, lw=3, zorder=1)
ax.scatter([r["y2019"] for r in rows], ys, s=45, facecolor=PAPER, edgecolor=MUTED, linewidths=2, zorder=2, label="2019")
ax.scatter([r["y2025"] for r in rows], ys, s=60, color=S[0], edgecolor=PAPER, linewidths=1.2, zorder=3, label="2025")
for y, r in zip(ys, rows):
    ax.text(r["y2025"] + (0.012 if r["y2025"] >= r["y2019"] else -0.012), y, f"{r['y2025'] * 100:.0f}%",
            va="center", ha="left" if r["y2025"] >= r["y2019"] else "right", fontsize=10.5)
ax.set_yticks(list(ys), [r["name"] for r in rows], fontsize=11.5)
ax.set_ylim(len(rows) - 0.5, -0.7)
ax.set_xlim(0, 0.6)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(pct))
ax.grid(True, axis="x")
ax.grid(False, axis="y")
ax.tick_params(length=0)
ax.legend(loc="lower right", bbox_to_anchor=(1, 1.0), ncol=2, fontsize=11)
finish(fig, "skills_by_director", "Keyword search of the companies' own skill lists. Annual-meeting proxy statements of 2019 and 2025.")

---
Code: MIT license. Data: [CC BY-NC 4.0](https://corporatedisclosuredata.org/data/LICENSE).
To cite the data, see [corporatedisclosuredata.org/#cite](https://corporatedisclosuredata.org/#cite). Questions about the data:
data@corporatedisclosuredata.org.